# Fill the median/MAD row — bottle defect localisation

Generated by `scripts/make_kaggle_kernel.py`. Do not edit here; edit the repo and
regenerate, otherwise the Kaggle run and the repo drift apart.

Trains the v3 autoencoder and evaluates it on the clean and lighting-stressed test
sets, scoring both ways (raw top-k, and median/MAD-normalised) in one pass over
identical anomaly maps.

**Settings this notebook needs:** GPU accelerator, Internet on (for one `pip install`),
and the `ipythonx/mvtec-ad` dataset attached.


In [ ]:
!pip install -q pytorch-msssim
import torch, sys
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## 1. Write the package

The repo's modules, embedded verbatim at generation time.


In [ ]:
import os, base64, pathlib
os.makedirs('/kaggle/working/defectloc', exist_ok=True)
FILES = {
    'defectloc/__init__.py': 'IiIiUmVjb25zdHJ1Y3Rpb24tYmFzZWQgZGVmZWN0IGxvY2FsaXNhdGlvbiBmb3IgaW5kdXN0cmlhbCBpbnNwZWN0aW9uLgoKQSBjb252b2x1dGlvbmFsIGF1dG9lbmNvZGVyIHRyYWluZWQgb25seSBvbiBub3JtYWwgYm90dGxlcyBwcm9kdWNlcyBhIHNwYXRpYWwKYW5vbWFseSBtYXAsIHNvIGEgZGVmZWN0J3MgbG9jYXRpb24gY2FuIGJlIHRyYWNlZCBiYWNrIHRvIHRoZSB1cHN0cmVhbQptYW51ZmFjdHVyaW5nIHN0YXRpb24gdGhhdCBjYXVzZWQgaXQuCgpUaGlzIHBhY2thZ2UgaXMgdGhlIG1vZHVsZSByZWZhY3RvciBvZiBgcGF0Y2hjb3JlLWJhc2UuaXB5bmJgICh0aGUgdjMgS2FnZ2xlCnByb3RvdHlwZSkuIFRoZSBudW1lcmljcyBhcmUgYSBmYWl0aGZ1bCBwb3J0OiBzYW1lIGFyY2hpdGVjdHVyZSwgc2FtZSBsb3NzLApzYW1lIGFub21hbHkgbWFwLCBzYW1lIGltYWdlIHNjb3JlLCBzbyB0aGUgbm90ZWJvb2sncyByZXN1bHRzIHJlcHJvZHVjZS4KIiIiCgpfX3ZlcnNpb25fXyA9ICIzLjEuMCIKCmZyb20gZGVmZWN0bG9jLmNvbmZpZyBpbXBvcnQgQ29uZmlnICAjIG5vcWE6IEY0MDEK',
    'defectloc/config.py': 'IiIiU2luZ2xlIHBsYWNlIGZvciB0aGUgdmFsdWVzIHRoYXQgZGVmaW5lIGEgcnVuLgoKVGhlIG5vdGVib29rIHNjYXR0ZXJlZCB0aGVzZSBhY3Jvc3MgY2VsbHMgYXMgbW9kdWxlLWxldmVsIGdsb2JhbHMuIENvbGxlY3RpbmcKdGhlbSBoZXJlIGlzIHdoYXQgbWFrZXMgYSBydW4gcmVwcm9kdWNpYmxlOiBvbmUgb2JqZWN0LCBsb2dnZWQgdmVyYmF0aW0gaW50bwp0aGUgcmVzdWx0cyBKU09OIHNvIGFueSBudW1iZXIgaW4gdGhlIHRhYmxlIGNhbiBiZSB0cmFjZWQgdG8gdGhlIHNldHRpbmdzIHRoYXQKcHJvZHVjZWQgaXQuCiIiIgoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBhc2RpY3QsIGZpZWxkCmZyb20gdHlwaW5nIGltcG9ydCBBbnksIERpY3QKCgojIFRoZSBub3RlYm9vaydzIGNvbnN0YW50cywgdW5jaGFuZ2VkLiBFZGl0aW5nIHRoZXNlIGNoYW5nZXMgdGhlIG51bWJlcnMsIHNvCiMgdGhleSBhcmUgZGVmYXVsdHMgcmF0aGVyIHRoYW4gbGl0ZXJhbHMgYnVyaWVkIGluIGZ1bmN0aW9uIGJvZGllcy4KSU1HX1NJWkUgPSAyNTYKCgpAZGF0YWNsYXNzCmNsYXNzIENvbmZpZzoKICAgICMgLS0tIGRhdGEgLS0tCiAgICBpbWdfc2l6ZTogaW50ID0gSU1HX1NJWkUKICAgIG51bV93b3JrZXJzOiBpbnQgPSAwICAgICAgICAgICMgMCBpcyB0aGUgc2FmZSBkZWZhdWx0IG9uIFdpbmRvd3MKCiAgICAjIC0tLSBtb2RlbCAtLS0KICAgIGxhdGVudF9jaDogaW50ID0gNjQgICAgICAgICAgICMgYm90dGxlbmVjayA2NHg4eDggPSA0MDk2IHZhbHVlcyAodjIgZml4KQoKICAgICMgLS0tIHRyYWluaW5nIC0tLQogICAgZXBvY2hzOiBpbnQgPSA4MAogICAgYmF0Y2hfc2l6ZTogaW50ID0gMTYKICAgIGxyOiBmbG9hdCA9IDJlLTQKICAgIG5vaXNlX3NpZ21hOiBmbG9hdCA9IDAuMSAgICAgICMgZGVub2lzaW5nIG9iamVjdGl2ZSAodjIgZml4KQogICAgbG9zc19hbHBoYTogZmxvYXQgPSAwLjUgICAgICAgIyAwLjUqTVNFICsgMC41KigxIC0gU1NJTSkKCiAgICAjIC0tLSBhbm9tYWx5IG1hcCAvIHNjb3JpbmcgLS0tCiAgICBzc2ltX3dpbmRvdzogaW50ID0gMTEgICAgICAgICAjIGxvY2FsLVNTSU0gZGlzc2ltaWxhcml0eSBtYXAgKHYzIGZpeCkKICAgIHRvcGtfZnJhYzogZmxvYXQgPSAwLjAxICAgICAgICMgaW1hZ2Ugc2NvcmUgPSBtZWFuIG9mIGhvdHRlc3QgMSUgb2YgcGl4ZWxzCgogICAgIyAtLS0gcmVwcm9kdWNpYmlsaXR5IC0tLQogICAgc2VlZDogaW50ID0gMAoKICAgIGRlZiB0b19kaWN0KHNlbGYpIC0+IERpY3Rbc3RyLCBBbnldOgogICAgICAgIHJldHVybiBhc2RpY3Qoc2VsZikK',
    'defectloc/model.py': 'IiIiVGhlIGF1dG9lbmNvZGVyLgoKU3ltbWV0cmljIDUtc3RhZ2UgY29udm9sdXRpb25hbCBhdXRvZW5jb2Rlci4gVGhlIGJvdHRsZW5lY2sgaXMgZGVsaWJlcmF0ZWx5CnRpZ2h0OiB2MSB1c2VkIDI1NngxNngxNiAoNjUsNTM2IHZhbHVlcykgYW5kIHRoZSBtb2RlbCBzaW1wbHkgY29waWVkIGl0cyBpbnB1dAp0aHJvdWdoLCBzbyBkZWZlY3RzIHN1cnZpdmVkIGludG8gdGhlIHJlY29uc3RydWN0aW9uIGFuZCBuZXZlciBzdG9vZCBvdXQuIFRoZQo2NHg4eDggYm90dGxlbmVjayBoZXJlIGlzIDE2eCBzbWFsbGVyIGFuZCBmb3JjZXMgdGhlIG1vZGVsIHRvIGxlYXJuIHRoZQoqc3RydWN0dXJlKiBvZiBhIG5vcm1hbCBib3R0bGUgaW5zdGVhZC4KIiIiCgppbXBvcnQgdG9yY2gubm4gYXMgbm4KCgpkZWYgX2VuY19ibG9jayhpbl9jaDogaW50LCBvdXRfY2g6IGludCkgLT4gbm4uU2VxdWVudGlhbDoKICAgIHJldHVybiBubi5TZXF1ZW50aWFsKAogICAgICAgIG5uLkNvbnYyZChpbl9jaCwgb3V0X2NoLCA0LCBzdHJpZGU9MiwgcGFkZGluZz0xKSwKICAgICAgICBubi5CYXRjaE5vcm0yZChvdXRfY2gpLAogICAgICAgIG5uLlJlTFUoaW5wbGFjZT1UcnVlKSwKICAgICkKCgpkZWYgX2RlY19ibG9jayhpbl9jaDogaW50LCBvdXRfY2g6IGludCwgbGFzdDogYm9vbCA9IEZhbHNlKSAtPiBubi5TZXF1ZW50aWFsOgogICAgbGF5ZXJzID0gW25uLkNvbnZUcmFuc3Bvc2UyZChpbl9jaCwgb3V0X2NoLCA0LCBzdHJpZGU9MiwgcGFkZGluZz0xKV0KICAgIGlmIGxhc3Q6CiAgICAgICAgbGF5ZXJzLmFwcGVuZChubi5TaWdtb2lkKCkpCiAgICBlbHNlOgogICAgICAgIGxheWVycyArPSBbbm4uQmF0Y2hOb3JtMmQob3V0X2NoKSwgbm4uUmVMVShpbnBsYWNlPVRydWUpXQogICAgcmV0dXJuIG5uLlNlcXVlbnRpYWwoKmxheWVycykKCgpjbGFzcyBDb252QXV0b2VuY29kZXIobm4uTW9kdWxlKToKICAgICIiIjN4MjU2eDI1NiAtPiA2NHg4eDggLT4gM3gyNTZ4MjU2LiIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBsYXRlbnRfY2g6IGludCA9IDY0KToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmxhdGVudF9jaCA9IGxhdGVudF9jaAogICAgICAgIHNlbGYuZW5jb2RlciA9IG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgIF9lbmNfYmxvY2soMywgMzIpLAogICAgICAgICAgICBfZW5jX2Jsb2NrKDMyLCA2NCksCiAgICAgICAgICAgIF9lbmNfYmxvY2soNjQsIDEyOCksCiAgICAgICAgICAgIF9lbmNfYmxvY2soMTI4LCAyNTYpLAogICAgICAgICAgICBfZW5jX2Jsb2NrKDI1NiwgbGF0ZW50X2NoKSwKICAgICAgICApCiAgICAgICAgc2VsZi5kZWNvZGVyID0gbm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgX2RlY19ibG9jayhsYXRlbnRfY2gsIDI1NiksCiAgICAgICAgICAgIF9kZWNfYmxvY2soMjU2LCAxMjgpLAogICAgICAgICAgICBfZGVjX2Jsb2NrKDEyOCwgNjQpLAogICAgICAgICAgICBfZGVjX2Jsb2NrKDY0LCAzMiksCiAgICAgICAgICAgIF9kZWNfYmxvY2soMzIsIDMsIGxhc3Q9VHJ1ZSksCiAgICAgICAgKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgpOgogICAgICAgIHJldHVybiBzZWxmLmRlY29kZXIoc2VsZi5lbmNvZGVyKHgpKQo=',
    'defectloc/losses.py': 'IiIiVHJhaW5pbmcgb2JqZWN0aXZlOiAwLjUqTVNFICsgMC41KigxIC0gU1NJTSkuCgpBIHRpZ2h0IGJvdHRsZW5lY2sgcHJvZHVjZXMgYmx1cnJ5IHJlY29uc3RydWN0aW9ucy4gUHVyZSBNU0UgcHVuaXNoZXMgdGhhdApibHVyIGV2ZXJ5d2hlcmUgYW5kIGRyYWdzIHRoZSBtb2RlbCB0b3dhcmQgY29weWluZzsgdGhlIFNTSU0gdGVybSB0b2xlcmF0ZXMKdW5pZm9ybSBibHVyIHdoaWxlIHN0aWxsIHBlbmFsaXNpbmcgKnN0cnVjdHVyYWwqIGRpZmZlcmVuY2UsIHdoaWNoIGlzIHdoYXQgYQpkZWZlY3QgaXMuCiIiIgoKaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgpmcm9tIHB5dG9yY2hfbXNzc2ltIGltcG9ydCBzc2ltIGFzIHNzaW1fZm4KCgpkZWYgcmVjb25fbG9zcyhyZWNvbiwgdGFyZ2V0LCBhbHBoYTogZmxvYXQgPSAwLjUpOgogICAgbXNlID0gRi5tc2VfbG9zcyhyZWNvbiwgdGFyZ2V0KQogICAgcyA9IHNzaW1fZm4ocmVjb24sIHRhcmdldCwgZGF0YV9yYW5nZT0xLjAsIHNpemVfYXZlcmFnZT1UcnVlKQogICAgcmV0dXJuIGFscGhhICogbXNlICsgKDEgLSBhbHBoYSkgKiAoMSAtIHMpCg==',
    'defectloc/data.py': 'IiIiRGF0YXNldHMgb3ZlciBhbiBNVlRlYy1BRCBjYXRlZ29yeSBmb2xkZXIuCgpFeHBlY3RlZCBsYXlvdXQgKHRoaXMgaXMgTVZUZWMgQUQncyBvd24gbGF5b3V0LCB1bmNoYW5nZWQpOgoKICAgIDxyb290Pi8KICAgICAgICB0cmFpbi9nb29kLyoucG5nCiAgICAgICAgdGVzdC9nb29kLyoucG5nCiAgICAgICAgdGVzdC88ZGVmZWN0X3R5cGU+LyoucG5nCiAgICAgICAgZ3JvdW5kX3RydXRoLzxkZWZlY3RfdHlwZT4vPHN0ZW0+X21hc2sucG5nCiIiIgoKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBMaXN0LCBPcHRpb25hbCwgVHVwbGUKCmltcG9ydCBjdjIKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaApmcm9tIHRvcmNoLnV0aWxzLmRhdGEgaW1wb3J0IERhdGFzZXQKCmZyb20gZGVmZWN0bG9jLmNvbmZpZyBpbXBvcnQgSU1HX1NJWkUKCgpkZWYgbG9hZF9pbWcocGF0aCwgaW1nX3NpemU6IGludCA9IElNR19TSVpFKSAtPiBucC5uZGFycmF5OgogICAgIiIiUmVhZCBhbiBpbWFnZSBhcyBmbG9hdDMyIFJHQiBpbiBbMCwgMV0sIEhXQy4iIiIKICAgIGltZyA9IGN2Mi5pbXJlYWQoc3RyKHBhdGgpKQogICAgaWYgaW1nIGlzIE5vbmU6CiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJjb3VsZCBub3QgcmVhZCBpbWFnZToge3BhdGh9IikKICAgIGltZyA9IGN2Mi5jdnRDb2xvcihpbWcsIGN2Mi5DT0xPUl9CR1IyUkdCKQogICAgaW1nID0gY3YyLnJlc2l6ZShpbWcsIChpbWdfc2l6ZSwgaW1nX3NpemUpKQogICAgcmV0dXJuIGltZy5hc3R5cGUobnAuZmxvYXQzMikgLyAyNTUuMAoKCmRlZiBsb2FkX21hc2socGF0aDogT3B0aW9uYWxbUGF0aF0sIGltZ19zaXplOiBpbnQgPSBJTUdfU0laRSkgLT4gbnAubmRhcnJheToKICAgICIiIlJlYWQgYSBiaW5hcnkgZ3JvdW5kLXRydXRoIG1hc2ssIG9yIGFuIGFsbC16ZXJvIG1hc2sgZm9yIGEgZ29vZCBpbWFnZS4iIiIKICAgIGlmIHBhdGggaXMgTm9uZSBvciBub3QgUGF0aChwYXRoKS5leGlzdHMoKToKICAgICAgICByZXR1cm4gbnAuemVyb3MoKGltZ19zaXplLCBpbWdfc2l6ZSksIG5wLmZsb2F0MzIpCiAgICBtID0gY3YyLmltcmVhZChzdHIocGF0aCksIGN2Mi5JTVJFQURfR1JBWVNDQUxFKQogICAgbSA9IGN2Mi5yZXNpemUobSwgKGltZ19zaXplLCBpbWdfc2l6ZSksIGludGVycG9sYXRpb249Y3YyLklOVEVSX05FQVJFU1QpCiAgICByZXR1cm4gKG0gPiAwKS5hc3R5cGUobnAuZmxvYXQzMikKCgpjbGFzcyBUcmFpbkdvb2QoRGF0YXNldCk6CiAgICAiIiJOb3JtYWwgaW1hZ2VzIG9ubHkgLS0gdGhlIG1vZGVsIG5ldmVyIHNlZXMgYSBkZWZlY3QgZHVyaW5nIHRyYWluaW5nLiIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCByb290LCBpbWdfc2l6ZTogaW50ID0gSU1HX1NJWkUpOgogICAgICAgIHNlbGYuaW1nX3NpemUgPSBpbWdfc2l6ZQogICAgICAgIHNlbGYucGF0aHM6IExpc3RbUGF0aF0gPSBzb3J0ZWQoKFBhdGgocm9vdCkgLyAidHJhaW4iIC8gImdvb2QiKS5nbG9iKCIqLnBuZyIpKQogICAgICAgIGlmIG5vdCBzZWxmLnBhdGhzOgogICAgICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcigKICAgICAgICAgICAgICAgIGYibm8gdHJhaW5pbmcgaW1hZ2VzIHVuZGVyIHtQYXRoKHJvb3QpIC8gJ3RyYWluJyAvICdnb29kJ30uICIKICAgICAgICAgICAgICAgICJSdW4gYHB5dGhvbiBzY3JpcHRzL3ByZXBhcmVfZGF0YS5weWAgZmlyc3QuIgogICAgICAgICAgICApCgogICAgZGVmIF9fbGVuX18oc2VsZikgLT4gaW50OgogICAgICAgIHJldHVybiBsZW4oc2VsZi5wYXRocykKCiAgICBkZWYgX19nZXRpdGVtX18oc2VsZiwgaTogaW50KSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgaW1nID0gbG9hZF9pbWcoc2VsZi5wYXRoc1tpXSwgc2VsZi5pbWdfc2l6ZSkKICAgICAgICByZXR1cm4gdG9yY2guZnJvbV9udW1weShpbWcpLnBlcm11dGUoMiwgMCwgMSkgICMgQ0hXCgoKY2xhc3MgVGVzdFNldChEYXRhc2V0KToKICAgICIiIkFsbCB0ZXN0IGltYWdlcywgdGhlaXIgZ3JvdW5kLXRydXRoIG1hc2tzLCBhbmQgdGhlIGltYWdlLWxldmVsIGxhYmVsLiIiIgoKICAgICMgTm90IGEgcHl0ZXN0IHRlc3QgY2xhc3MsIGRlc3BpdGUgdGhlIG5hbWU7IHRoZSBuYW1lIG1hdGNoZXMgTVZUZWMncyBzcGxpdC4KICAgIF9fdGVzdF9fID0gRmFsc2UKCiAgICBkZWYgX19pbml0X18oc2VsZiwgcm9vdCwgaW1nX3NpemU6IGludCA9IElNR19TSVpFKToKICAgICAgICByb290ID0gUGF0aChyb290KQogICAgICAgIHNlbGYuaW1nX3NpemUgPSBpbWdfc2l6ZQogICAgICAgIHNlbGYuaXRlbXM6IExpc3RbVHVwbGVbUGF0aCwgT3B0aW9uYWxbUGF0aF0sIGludF1dID0gW10KICAgICAgICB0ZXN0X2RpciA9IHJvb3QgLyAidGVzdCIKICAgICAgICBpZiBub3QgdGVzdF9kaXIuaXNfZGlyKCk6CiAgICAgICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKAogICAgICAgICAgICAgICAgZiJubyB0ZXN0IHNwbGl0IHVuZGVyIHt0ZXN0X2Rpcn0uICIKICAgICAgICAgICAgICAgICJSdW4gYHB5dGhvbiBzY3JpcHRzL3ByZXBhcmVfZGF0YS5weWAgZmlyc3QuIgogICAgICAgICAgICApCiAgICAgICAgZm9yIHN1YiBpbiBzb3J0ZWQodGVzdF9kaXIuaXRlcmRpcigpKToKICAgICAgICAgICAgaWYgbm90IHN1Yi5pc19kaXIoKToKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIGRlZmVjdCA9IHN1Yi5uYW1lICAjICdnb29kJyBvciBhIGRlZmVjdCB0eXBlCiAgICAgICAgICAgIGZvciBwIGluIHNvcnRlZChzdWIuZ2xvYigiKi5wbmciKSk6CiAgICAgICAgICAgICAgICBsYWJlbCA9IDAgaWYgZGVmZWN0ID09ICJnb29kIiBlbHNlIDEKICAgICAgICAgICAgICAgIG1hc2sgPSAoCiAgICAgICAgICAgICAgICAgICAgTm9uZQogICAgICAgICAgICAgICAgICAgIGlmIGRlZmVjdCA9PSAiZ29vZCIKICAgICAgICAgICAgICAgICAgICBlbHNlIHJvb3QgLyAiZ3JvdW5kX3RydXRoIiAvIGRlZmVjdCAvIGYie3Auc3RlbX1fbWFzay5wbmciCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICBzZWxmLml0ZW1zLmFwcGVuZCgocCwgbWFzaywgbGFiZWwpKQogICAgICAgIGlmIG5vdCBzZWxmLml0ZW1zOgogICAgICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmInt0ZXN0X2Rpcn0gY29udGFpbnMgbm8gLnBuZyBpbWFnZXMiKQoKICAgIGRlZiBfX2xlbl9fKHNlbGYpIC0+IGludDoKICAgICAgICByZXR1cm4gbGVuKHNlbGYuaXRlbXMpCgogICAgZGVmIF9fZ2V0aXRlbV9fKHNlbGYsIGk6IGludCk6CiAgICAgICAgcGF0aCwgbWFzaywgbGFiZWwgPSBzZWxmLml0ZW1zW2ldCiAgICAgICAgaW1nID0gdG9yY2guZnJvbV9udW1weShsb2FkX2ltZyhwYXRoLCBzZWxmLmltZ19zaXplKSkucGVybXV0ZSgyLCAwLCAxKQogICAgICAgIG0gPSB0b3JjaC5mcm9tX251bXB5KGxvYWRfbWFzayhtYXNrLCBzZWxmLmltZ19zaXplKSkKICAgICAgICByZXR1cm4gaW1nLCBtLCBsYWJlbAo=',
    'defectloc/augment.py': 'IiIiVGhlIGZhY3RvcnktbGlnaHRpbmcgc3RyZXNzIHRlc3QuCgpBcHBsaWVkIHRvIHRoZSAqKnRlc3Qgc2V0IG9ubHkqKi4gVHJhaW5pbmcgc3RheXMgY2xlYW4sIHNvIHRoaXMgbWVhc3VyZXMgd2hhdApoYXBwZW5zIHdoZW4gYSBtb2RlbCB0cmFpbmVkIHVuZGVyIG9uZSBsaWdodGluZyByaWcgbWVldHMgYW5vdGhlciAtLSB0aGUgc2luZ2xlCm1vc3QgY29tbW9uIHdheSBhIHdvcmtpbmcgaW5zcGVjdGlvbiBtb2RlbCBkZWdyYWRlcyBhZnRlciBpbnN0YWxsYXRpb24uCgpSYW5nZXMgYXJlIGNob3NlbiB0byBiZSBub3RpY2VhYmxlIGJ1dCBub3QgZGVzdHJ1Y3RpdmU6IGEgcGVyc29uIHNob3VsZCBzdGlsbAplYXNpbHkgc2VlIHRoZSBib3R0bGUgYW5kIGFueSBkZWZlY3QuIElmIGEgaHVtYW4gY2Fubm90LCB0aGUgbW9kZWwgc2hvdWxkIG5vdApiZSBibGFtZWQgZm9yIGZhaWxpbmcgYW5kIHRoZSB0ZXN0IGlzIHVuZmFpci4KIiIiCgppbXBvcnQgcmFuZG9tCgppbXBvcnQgYWxidW1lbnRhdGlvbnMgYXMgQQppbXBvcnQgbnVtcHkgYXMgbnAKCgpjbGFzcyBEaXJlY3Rpb25hbFNoYWRvdyhBLkltYWdlT25seVRyYW5zZm9ybSk6CiAgICAiIiJEYXJrZW4gdGhlIGltYWdlIGFsb25nIGEgcmFuZG9tIGxpbmVhciBncmFkaWVudC4KCiAgICBNaW1pY3MgdW5ldmVuIG92ZXJoZWFkIGxpZ2h0aW5nIG9yIGEgc2hhZG93IGNhc3QgYWNyb3NzIHRoZSBjb252ZXlvciAtLQogICAgdGhlIHByb2R1Y3Rpb24tc3BlY2lmaWMgZmFpbHVyZSBtb2RlIHRoYXQgYSBnbG9iYWwgYnJpZ2h0bmVzcyBzaGlmdCBkb2VzCiAgICBub3QgY2FwdHVyZS4KICAgICIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBtaW5fZmFjdG9yOiBmbG9hdCA9IDAuNCwgbWF4X2ZhY3RvcjogZmxvYXQgPSAwLjg1LCBwOiBmbG9hdCA9IDAuNSk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXyhwPXApCiAgICAgICAgc2VsZi5taW5fZmFjdG9yID0gbWluX2ZhY3RvciAgIyBkYXJrZXN0IG11bHRpcGxpZXIgYXQgdGhlIHNoYWRvd2VkIGVkZ2UKICAgICAgICBzZWxmLm1heF9mYWN0b3IgPSBtYXhfZmFjdG9yCgogICAgZGVmIGFwcGx5KHNlbGYsIGltZywgKiprd2FyZ3MpOgogICAgICAgIGgsIHcgPSBpbWcuc2hhcGVbOjJdCiAgICAgICAgYW5nbGUgPSBucC5yYW5kb20udW5pZm9ybSgwLCAyICogbnAucGkpCiAgICAgICAgeHgsIHl5ID0gbnAubWVzaGdyaWQobnAubGluc3BhY2UoMCwgMSwgdyksIG5wLmxpbnNwYWNlKDAsIDEsIGgpKQogICAgICAgIGdyYWQgPSB4eCAqIG5wLmNvcyhhbmdsZSkgKyB5eSAqIG5wLnNpbihhbmdsZSkKICAgICAgICBncmFkID0gKGdyYWQgLSBncmFkLm1pbigpKSAvIChncmFkLm1heCgpIC0gZ3JhZC5taW4oKSArIDFlLTgpCiAgICAgICAgZGFyayA9IG5wLnJhbmRvbS51bmlmb3JtKHNlbGYubWluX2ZhY3Rvciwgc2VsZi5tYXhfZmFjdG9yKQogICAgICAgIG1hc2sgPSBkYXJrICsgKDEgLSBkYXJrKSAqIGdyYWQgICMgcmFuZ2VzIFtkYXJrLCAxLjBdIGFjcm9zcyB0aGUgaW1hZ2UKICAgICAgICBvdXQgPSBpbWcuYXN0eXBlKG5wLmZsb2F0MzIpICogbWFza1suLi4sIE5vbmVdCiAgICAgICAgcmV0dXJuIG5wLmNsaXAob3V0LCAwLCAyNTUpLmFzdHlwZShpbWcuZHR5cGUpCgogICAgZGVmIGdldF90cmFuc2Zvcm1faW5pdF9hcmdzX25hbWVzKHNlbGYpOgogICAgICAgIHJldHVybiAoIm1pbl9mYWN0b3IiLCAibWF4X2ZhY3RvciIpCgoKZGVmIGJ1aWxkX2ZhY3RvcnlfbGlnaHRpbmcoc2VlZDogaW50IHwgTm9uZSA9IE5vbmUpIC0+IEEuQ29tcG9zZToKICAgICIiIlBob3RvbWV0cmljLW9ubHkgcGlwZWxpbmU6IG1hc2tzIHBhc3MgdGhyb3VnaCB1bmNoYW5nZWQuCgogICAgVGhlIG5vdGVib29rIGxlZnQgdGhpcyB1bnNlZWRlZCwgd2hpY2ggbWVhbnQgdGhlIGxpZ2h0aW5nIHRlc3Qgc2V0IHdhcyBhCiAgICBkaWZmZXJlbnQgc2V0IG9mIGltYWdlcyBvbiBldmVyeSBydW4gYW5kIGl0cyBudW1iZXJzIHdlcmUgbm90IGNvbXBhcmFibGUKICAgIGJldHdlZW4gcnVucy4gU2VlZGluZyBpdCBpcyB0aGUgZGlmZmVyZW5jZSBiZXR3ZWVuIGEgc3RyZXNzIHRlc3QgYW5kIGFuCiAgICBhbmVjZG90ZS4KICAgICIiIgogICAgdHJhbnNmb3JtcyA9IFsKICAgICAgICAjIE92ZXJhbGwgYnJpZ2h0bmVzcy9jb250cmFzdCBkcmlmdCAtLSBidWxicyBkaW1taW5nLCBoYXplLgogICAgICAgIEEuUmFuZG9tQnJpZ2h0bmVzc0NvbnRyYXN0KAogICAgICAgICAgICBicmlnaHRuZXNzX2xpbWl0PSgtMC4yNSwgMC4yNSksICAgIyArLy0yNSUsIHZpc2libGUgYnV0IG5vdCBibG93biBvdXQKICAgICAgICAgICAgY29udHJhc3RfbGltaXQ9KC0wLjIwLCAwLjIwKSwgICAgICMgKy8tMjAlCiAgICAgICAgICAgIHA9MC45LAogICAgICAgICksCiAgICAgICAgIyBOb24tbGluZWFyIGV4cG9zdXJlIHJlc3BvbnNlLCBtb3JlIHJlYWxpc3RpYyB0aGFuIGxpbmVhciBicmlnaHRuZXNzLgogICAgICAgIEEuUmFuZG9tR2FtbWEoZ2FtbWFfbGltaXQ9KDgwLCAxMjApLCBwPTAuNyksICAgIyAwLjh4IC0gMS4yeCBnYW1tYQogICAgICAgICMgQ29sb3VyIHRlbXBlcmF0dXJlOiB3YXJtL2Nvb2wgY2FzdCBmcm9tIGRpZmZlcmVudCBidWxiIHR5cGVzLiBNaWxkIC0tCiAgICAgICAgIyBnbGFzcyB0aW50IHNob3VsZCBzaGlmdCB3aXRob3V0IHJlY29sb3VyaW5nIHRoZSBib3R0bGUgZW50aXJlbHkuCiAgICAgICAgQS5IdWVTYXR1cmF0aW9uVmFsdWUoCiAgICAgICAgICAgIGh1ZV9zaGlmdF9saW1pdD04LAogICAgICAgICAgICBzYXRfc2hpZnRfbGltaXQ9MTUsCiAgICAgICAgICAgIHZhbF9zaGlmdF9saW1pdD0xMCwKICAgICAgICAgICAgcD0wLjYsCiAgICAgICAgKSwKICAgICAgICBEaXJlY3Rpb25hbFNoYWRvdyhtaW5fZmFjdG9yPTAuNDUsIG1heF9mYWN0b3I9MC44NSwgcD0wLjUpLAogICAgXQogICAgaWYgc2VlZCBpcyBub3QgTm9uZToKICAgICAgICByYW5kb20uc2VlZChzZWVkKQogICAgICAgIG5wLnJhbmRvbS5zZWVkKHNlZWQpCiAgICB0cnk6CiAgICAgICAgIyBhbGJ1bWVudGF0aW9ucyA+PSAxLjQuMjEgYWNjZXB0cyBhIHBlci1Db21wb3NlIHNlZWQuCiAgICAgICAgcmV0dXJuIEEuQ29tcG9zZSh0cmFuc2Zvcm1zLCBhZGRpdGlvbmFsX3RhcmdldHM9e30sIHNlZWQ9c2VlZCkKICAgIGV4Y2VwdCBUeXBlRXJyb3I6CiAgICAgICAgIyBPbGRlciByZWxlYXNlczogdGhlIGdsb2JhbCBzZWVkcyBzZXQgYWJvdmUgYXJlIHdoYXQgbWFrZSBpdCByZXBlYXRhYmxlLgogICAgICAgIHJldHVybiBBLkNvbXBvc2UodHJhbnNmb3JtcywgYWRkaXRpb25hbF90YXJnZXRzPXt9KQo=',
    'defectloc/anomaly.py': 'IiIiQW5vbWFseSBtYXAgYW5kIGltYWdlIHNjb3JlLgoKVHdvIHNlcGFyYXRlIGRlY2lzaW9ucyBsaXZlIGhlcmUsIGFuZCB0aGUgcHJvamVjdCdzIG9wZW4gYnVnIGNhbWUgZnJvbQpjb25mbGF0aW5nIHRoZW06CgoqIHRoZSAqKm1hcCoqIHNheXMgKndoZXJlKiB0aGUgaW1hZ2UgaXMgYW5vbWFsb3VzIChwaXhlbC1BVVJPQyksIGFuZAoqIHRoZSAqKnNjb3JlKiogcmVkdWNlcyB0aGF0IG1hcCB0byBvbmUgbnVtYmVyIHNheWluZyAqd2hldGhlciogdGhlIGltYWdlIGlzCiAgYW5vbWFsb3VzIChpbWFnZS1BVVJPQykuCgp2MyBmaXhlZCB0aGUgbWFwIGJ5IHN3aXRjaGluZyBmcm9tIHBpeGVsLU1TRSB0byBsb2NhbC1TU0lNIGRpc3NpbWlsYXJpdHkKKCsxNSBwb2ludHMgb2YgcGl4ZWwtQVVST0Mgd2l0aCBubyByZXRyYWluaW5nKS4gVGhlIG1hcCBzdXJ2aXZlZCB0aGUgbGlnaHRpbmcKc3RyZXNzIHRlc3Q7IHRoZSBzY29yZSBkaWQgbm90LiBTZWUgYGltYWdlX3Njb3JlYCBmb3Igd2h5LCBhbmQgZm9yIHRoZSBmaXguCiIiIgoKZnJvbSB0eXBpbmcgaW1wb3J0IExpdGVyYWwKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgoKU2NvcmVOb3JtID0gTGl0ZXJhbFsibm9uZSIsICJtZWRpYW5fbWFkIl0KCiMgU3RhbmRhcmQgU1NJTSBzdGFiaWxpc2VycyBmb3IgZGF0YSBpbiBbMCwgMV0uCl9DMSA9IDAuMDEgKiogMgpfQzIgPSAwLjAzICoqIDIKCgpkZWYgbG9jYWxfc3NpbV9kaXNzaW1pbGFyaXR5KHg6IHRvcmNoLlRlbnNvciwgeTogdG9yY2guVGVuc29yLCB3aW46IGludCA9IDExKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAiIiJQZXItcGl4ZWwgc3RydWN0dXJhbCBkaXNzaW1pbGFyaXR5IGJldHdlZW4gdHdvIGJhdGNoZXMuIEhpZ2hlciA9IG1vcmUgYW5vbWFsb3VzLgoKICAgIFdpbmRvd2VkIFNTSU0gY29tcHV0ZWQgb24gdGhlIGNoYW5uZWwtYXZlcmFnZWQgaW1hZ2UsIHJldHVybmVkIGFzIGBgMSAtIFNTSU1gYAogICAgYW5kIGNsYW1wZWQgdG8gWzAsIDFdLiBSZWFkaW5nICpzdHJ1Y3R1cmUqIHJhdGhlciB0aGFuIHJhdyBwaXhlbCB2YWx1ZXMgaXMKICAgIHdoYXQgc3RvcHMgdGhlIG1hcCBvdmVyLWZpcmluZyBvbiBub3JtYWwgaGlnaC1mcmVxdWVuY3kgcmVnaW9ucyAtLSByaW1zIGFuZAogICAgcmVmbGVjdGlvbnMsIHdoaWNoIHBpeGVsLU1TRSBsaXQgdXAgYXMgYnJpZ2h0bHkgYXMgcmVhbCBkZWZlY3RzLgogICAgIiIiCiAgICB4ZyA9IHgubWVhbigxLCBrZWVwZGltPVRydWUpCiAgICB5ZyA9IHkubWVhbigxLCBrZWVwZGltPVRydWUpCiAgICBwYWQgPSB3aW4gLy8gMgoKICAgIG11X3ggPSBGLmF2Z19wb29sMmQoeGcsIHdpbiwgMSwgcGFkKQogICAgbXVfeSA9IEYuYXZnX3Bvb2wyZCh5Zywgd2luLCAxLCBwYWQpCiAgICBtdV94MiwgbXVfeTIsIG11X3h5ID0gbXVfeCAqIG11X3gsIG11X3kgKiBtdV95LCBtdV94ICogbXVfeQoKICAgIHNpZ194ID0gRi5hdmdfcG9vbDJkKHhnICogeGcsIHdpbiwgMSwgcGFkKSAtIG11X3gyCiAgICBzaWdfeSA9IEYuYXZnX3Bvb2wyZCh5ZyAqIHlnLCB3aW4sIDEsIHBhZCkgLSBtdV95MgogICAgc2lnX3h5ID0gRi5hdmdfcG9vbDJkKHhnICogeWcsIHdpbiwgMSwgcGFkKSAtIG11X3h5CgogICAgc3NpbV9tYXAgPSAoKDIgKiBtdV94eSArIF9DMSkgKiAoMiAqIHNpZ194eSArIF9DMikpIC8gKAogICAgICAgIChtdV94MiArIG11X3kyICsgX0MxKSAqIChzaWdfeCArIHNpZ195ICsgX0MyKQogICAgKQogICAgcmV0dXJuICgxIC0gc3NpbV9tYXApLmNsYW1wKDAsIDEpLnNxdWVlemUoMSkgICMgQixILFcKCgpAdG9yY2gubm9fZ3JhZCgpCmRlZiBhbm9tYWx5X21hcChtb2RlbCwgeDogdG9yY2guVGVuc29yLCB3aW46IGludCA9IDExKSAtPiBucC5uZGFycmF5OgogICAgIiIiUmVjb25zdHJ1Y3QgYGB4YGAgYW5kIHJldHVybiB0aGUgQixILFcgZGlzc2ltaWxhcml0eSBtYXAgYXMgbnVtcHkuIiIiCiAgICBtb2RlbC5ldmFsKCkKICAgIHJlY29uID0gbW9kZWwoeCkKICAgIHJldHVybiBsb2NhbF9zc2ltX2Rpc3NpbWlsYXJpdHkoeCwgcmVjb24sIHdpbj13aW4pLmNwdSgpLm51bXB5KCkKCgpkZWYgbm9ybWFsaXplX21hcChhbWFwOiBucC5uZGFycmF5LCBlcHM6IGZsb2F0ID0gMWUtOCkgLT4gbnAubmRhcnJheToKICAgICIiIlJvYnVzdCBwZXItaW1hZ2Ugc3RhbmRhcmRpc2F0aW9uOiBgYChhbWFwIC0gbWVkaWFuKSAvIE1BRGBgLgoKICAgIFRoaXMgaXMgdGhlIGZpeCBmb3IgdGhlIGxpZ2h0aW5nIGZhaWx1cmUuIEEgZ2xvYmFsIGxpZ2h0aW5nIHNoaWZ0IG1vdmVzIGFuCiAgICBpbWFnZSdzICp3aG9sZSogZXJyb3IgYmFzZWxpbmUgLS0gYSBkaW0gZnJhbWUgcmVjb25zdHJ1Y3RzIHdvcnNlIGV2ZXJ5d2hlcmUsCiAgICBzbyBldmVyeSBwaXhlbCBvZiBpdHMgbWFwIHNpdHMgaGlnaGVyLiBSYXcgdG9wLWsgc2NvcmVzIHRoZXJlZm9yZSBzdG9wIGJlaW5nCiAgICBjb21wYXJhYmxlIGJldHdlZW4gaW1hZ2VzIHNob3QgdW5kZXIgZGlmZmVyZW50IGxpZ2h0aW5nLCBhbmQgdGhlIGltYWdlLWxldmVsCiAgICByYW5raW5nIGNvbGxhcHNlcyBldmVuIHRob3VnaCB0aGUgbWFwIHN0aWxsIHBvaW50cyBhdCB0aGUgcmlnaHQgcGl4ZWxzLgoKICAgIE1lZGlhbiBhbmQgTUFEIGRlc2NyaWJlIHRoYXQgaW1hZ2UncyBvd24gbm9ybWFsIGJhY2tncm91bmQgKHRoZSBkZWZlY3QgaXMgYQogICAgZmV3IHBlcmNlbnQgb2YgdGhlIHBpeGVscywgc28gaXQgYmFyZWx5IG1vdmVzIGVpdGhlcikuIERpdmlkaW5nIGl0IG91dCBhc2tzCiAgICAiaG93IGZhciBhYm92ZSAqdGhpcyBpbWFnZSdzKiBiYWNrZ3JvdW5kIGlzIGl0cyBob3R0ZXN0IHJlZ2lvbiIsIHdoaWNoIGlzCiAgICB0aGUgcXVlc3Rpb24gdGhlIHNjb3JlIHdhcyBhbHdheXMgbWVhbnQgdG8gYW5zd2VyLgoKICAgIE1lZGlhbiBhbmQgTUFEIGFyZSB1c2VkIHJhdGhlciB0aGFuIG1lYW4gYW5kIHN0YW5kYXJkIGRldmlhdGlvbiBwcmVjaXNlbHkKICAgIGJlY2F1c2UgdGhlIGRlZmVjdCBpcyBhbiBvdXRsaWVyOiBpdCB3b3VsZCBpbmZsYXRlIHRoZSBtZWFuIGFuZCB0aGUgc3RhbmRhcmQKICAgIGRldmlhdGlvbiBpdCBpcyBzdXBwb3NlZCB0byBiZSBtZWFzdXJlZCBhZ2FpbnN0LgogICAgIiIiCiAgICBtZWQgPSBucC5tZWRpYW4oYW1hcCkKICAgIG1hZCA9IG5wLm1lZGlhbihucC5hYnMoYW1hcCAtIG1lZCkpCiAgICByZXR1cm4gKGFtYXAgLSBtZWQpIC8gKG1hZCArIGVwcykKCgpkZWYgaW1hZ2Vfc2NvcmUoYW1hcDogbnAubmRhcnJheSwgdG9wa19mcmFjOiBmbG9hdCA9IDAuMDEsIG5vcm06IFNjb3JlTm9ybSA9ICJub25lIikgLT4gZmxvYXQ6CiAgICAiIiJSZWR1Y2UgYW4gYW5vbWFseSBtYXAgdG8gb25lIG51bWJlcjogbWVhbiBvZiB0aGUgaG90dGVzdCBgYHRvcGtfZnJhY2BgIHBpeGVscy4KCiAgICBUaGUgdG9wLWsgbWVhbiBpcyBtb3JlIHJvYnVzdCB0aGFuIGEgcHVyZSBtYXgsIHdoaWNoIGEgc2luZ2xlIGhvdCBwaXhlbCBvZgogICAgcmVjb25zdHJ1Y3Rpb24gbm9pc2UgY2FuIGRvbWluYXRlLgoKICAgIE5vdGUgdGhhdCBgYG5vcm1gYCBjYW5ub3QgY2hhbmdlICp3aGljaCogcGl4ZWxzIGFyZSBzZWxlY3RlZCAtLSB0aGUKICAgIG5vcm1hbGlzYXRpb24gaXMgbW9ub3RvbmljIHdpdGhpbiBhbiBpbWFnZSwgc28gdGhlIHNhbWUgcGl4ZWxzIHN0YXkgaG90dGVzdC4KICAgIEl0IG9ubHkgY2hhbmdlcyB0aGUgdW5pdHMgdGhlIHNjb3JlIGlzIHJlcG9ydGVkIGluLCBhbmQgdGhhdCBpcyBleGFjdGx5IHRoZQogICAgcG9pbnQ6IGl0IG1ha2VzIHNjb3JlcyBjb21wYXJhYmxlICphY3Jvc3MqIGltYWdlcy4KICAgICIiIgogICAgaWYgbm9ybSA9PSAibWVkaWFuX21hZCI6CiAgICAgICAgYW1hcCA9IG5vcm1hbGl6ZV9tYXAoYW1hcCkKICAgIGVsaWYgbm9ybSAhPSAibm9uZSI6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmInVua25vd24gc2NvcmUgbm9ybWFsaXNhdGlvbjoge25vcm0hcn0iKQoKICAgIGZsYXQgPSBucC5zb3J0KGFtYXAucmF2ZWwoKSlbOjotMV0KICAgIGsgPSBtYXgoMSwgaW50KGxlbihmbGF0KSAqIHRvcGtfZnJhYykpCiAgICByZXR1cm4gZmxvYXQoZmxhdFs6a10ubWVhbigpKQo=',
    'defectloc/train.py': 'IiIiVHJhaW4gdGhlIGF1dG9lbmNvZGVyIG9uIG5vcm1hbCBpbWFnZXMgb25seS4KCiAgICBweXRob24gLW0gZGVmZWN0bG9jLnRyYWluIC0tZGF0YS1yb290IGRhdGEvTVZUZWNBRC9ib3R0bGUKIiIiCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGpzb24KaW1wb3J0IHJhbmRvbQppbXBvcnQgdGltZQppbXBvcnQgd2FybmluZ3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCmZyb20gdG9yY2gudXRpbHMuZGF0YSBpbXBvcnQgRGF0YUxvYWRlcgoKZnJvbSBkZWZlY3Rsb2MgaW1wb3J0IF9fdmVyc2lvbl9fCmZyb20gZGVmZWN0bG9jLmNvbmZpZyBpbXBvcnQgQ29uZmlnCmZyb20gZGVmZWN0bG9jLmRhdGEgaW1wb3J0IFRyYWluR29vZApmcm9tIGRlZmVjdGxvYy5sb3NzZXMgaW1wb3J0IHJlY29uX2xvc3MKZnJvbSBkZWZlY3Rsb2MubW9kZWwgaW1wb3J0IENvbnZBdXRvZW5jb2RlcgoKCmRlZiBzZXRfc2VlZChzZWVkOiBpbnQpIC0+IE5vbmU6CiAgICByYW5kb20uc2VlZChzZWVkKQogICAgbnAucmFuZG9tLnNlZWQoc2VlZCkKICAgIHRvcmNoLm1hbnVhbF9zZWVkKHNlZWQpCiAgICB0b3JjaC5jdWRhLm1hbnVhbF9zZWVkX2FsbChzZWVkKQoKCmRlZiBjdWRhX2lzX3VzYWJsZSgpIC0+IGJvb2w6CiAgICAiIiJXaGV0aGVyIENVREEgaXMgcHJlc2VudCBBTkQgdGhpcyB0b3JjaCBidWlsZCBjYW4gYWN0dWFsbHkgcnVuIG9uIHRoZSBHUFUuCgogICAgYHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKClgIGlzIG5vdCBlbm91Z2guIEl0IHJlcG9ydHMgVHJ1ZSBmb3IgYSBHUFUgd2hvc2UKICAgIGNvbXB1dGUgY2FwYWJpbGl0eSB0aGUgaW5zdGFsbGVkIHRvcmNoIHdhcyBub3QgY29tcGlsZWQgZm9yLCBhbmQgdGhlIGZhaWx1cmUKICAgIG9ubHkgc3VyZmFjZXMgbGF0ZXIgYXMgIm5vIGtlcm5lbCBpbWFnZSBpcyBhdmFpbGFibGUgZm9yIGV4ZWN1dGlvbiBvbiB0aGUKICAgIGRldmljZSIsIG1pZC10cmFpbmluZy4gS2FnZ2xlIGhpdHMgdGhpcyBleGFjdGx5OiBpdCBtYXkgaGFuZCBvdXQgYSBUZXNsYQogICAgUDEwMCAoc21fNjApIGFsb25nc2lkZSBhIHRvcmNoIGJ1aWxkIHN1cHBvcnRpbmcgc21fNzAgYW5kIHVwLgoKICAgIFNvIHJ1biBhIHJlYWwgY29udm9sdXRpb24gYW5kIHNlZS4gQSBwcm9iZSB0aGF0IGNvc3RzIG1pY3Jvc2Vjb25kcyBiZWF0cyBhCiAgICBjcmFzaCBhbiBob3VyIGludG8gYSBydW4uCiAgICAiIiIKICAgIGlmIG5vdCB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgIHJldHVybiBGYWxzZQogICAgdHJ5OgogICAgICAgIHggPSB0b3JjaC56ZXJvcygxLCAzLCA4LCA4LCBkZXZpY2U9ImN1ZGEiKQogICAgICAgIHcgPSB0b3JjaC56ZXJvcygzLCAzLCAzLCAzLCBkZXZpY2U9ImN1ZGEiKQogICAgICAgIHRvcmNoLm5uLmZ1bmN0aW9uYWwuY29udjJkKHgsIHcsIHBhZGRpbmc9MSkKICAgICAgICB0b3JjaC5jdWRhLnN5bmNocm9uaXplKCkKICAgICAgICByZXR1cm4gVHJ1ZQogICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOiAgIyBub3FhOiBCTEUwMDEgLSBhbnkgQ1VEQSBmYWlsdXJlIG1lYW5zIGZhbGwgYmFjawogICAgICAgIG5hbWUgPSAidW5rbm93biBHUFUiCiAgICAgICAgdHJ5OgogICAgICAgICAgICBuYW1lID0gdG9yY2guY3VkYS5nZXRfZGV2aWNlX25hbWUoMCkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOiAgIyBub3FhOiBCTEUwMDEKICAgICAgICAgICAgcGFzcwogICAgICAgIHdhcm5pbmdzLndhcm4oCiAgICAgICAgICAgIGYiQ1VEQSByZXBvcnRzIGF2YWlsYWJsZSBidXQgaXMgbm90IHVzYWJsZSBvbiB7bmFtZX0gKHt0eXBlKGUpLl9fbmFtZV9ffToge2V9KS4gIgogICAgICAgICAgICAiRmFsbGluZyBiYWNrIHRvIENQVS4gVHJhaW5pbmcgd2lsbCBiZSBtdWNoIHNsb3dlcjsgcGFzcyAtLWRldmljZSBjdWRhIHRvICIKICAgICAgICAgICAgIm92ZXJyaWRlIGFuZCBzZWUgdGhlIHJlYWwgZXJyb3IuIiwKICAgICAgICAgICAgUnVudGltZVdhcm5pbmcsCiAgICAgICAgICAgIHN0YWNrbGV2ZWw9MiwKICAgICAgICApCiAgICAgICAgcmV0dXJuIEZhbHNlCgoKZGVmIHBpY2tfZGV2aWNlKHJlcXVlc3RlZDogc3RyID0gImF1dG8iKSAtPiB0b3JjaC5kZXZpY2U6CiAgICAiIiJSZXNvbHZlIHRoZSBkZXZpY2UuICdhdXRvJyBmYWxscyBiYWNrIHRvIENQVSB3aGVuIHRoZSBHUFUgY2Fubm90IHJ1biBvdXIgb3BzLiIiIgogICAgaWYgcmVxdWVzdGVkICE9ICJhdXRvIjoKICAgICAgICByZXR1cm4gdG9yY2guZGV2aWNlKHJlcXVlc3RlZCkKICAgIHJldHVybiB0b3JjaC5kZXZpY2UoImN1ZGEiIGlmIGN1ZGFfaXNfdXNhYmxlKCkgZWxzZSAiY3B1IikKCgpkZWYgdHJhaW4ocm9vdCwgY2ZnOiBDb25maWcsIGRldmljZTogdG9yY2guZGV2aWNlLCBsb2dfZXZlcnk6IGludCA9IDEwLCB2ZXJib3NlOiBib29sID0gVHJ1ZSk6CiAgICAiIiJUcmFpbiBvbiBgYDxyb290Pi90cmFpbi9nb29kYGAgYW5kIHJldHVybiAobW9kZWwsIGhpc3RvcnkpLiIiIgogICAgc2V0X3NlZWQoY2ZnLnNlZWQpCgogICAgZHMgPSBUcmFpbkdvb2Qocm9vdCwgaW1nX3NpemU9Y2ZnLmltZ19zaXplKQogICAgbG9hZGVyID0gRGF0YUxvYWRlcigKICAgICAgICBkcywKICAgICAgICBiYXRjaF9zaXplPWNmZy5iYXRjaF9zaXplLAogICAgICAgIHNodWZmbGU9VHJ1ZSwKICAgICAgICBudW1fd29ya2Vycz1jZmcubnVtX3dvcmtlcnMsCiAgICAgICAgZHJvcF9sYXN0PWxlbihkcykgPiBjZmcuYmF0Y2hfc2l6ZSwKICAgICkKICAgIG1vZGVsID0gQ29udkF1dG9lbmNvZGVyKGxhdGVudF9jaD1jZmcubGF0ZW50X2NoKS50byhkZXZpY2UpCiAgICBvcHQgPSB0b3JjaC5vcHRpbS5BZGFtKG1vZGVsLnBhcmFtZXRlcnMoKSwgbHI9Y2ZnLmxyKQoKICAgIGhpc3RvcnkgPSBbXQogICAgbW9kZWwudHJhaW4oKQogICAgZm9yIGVwIGluIHJhbmdlKGNmZy5lcG9jaHMpOgogICAgICAgIHRvdGFsLCBuYiA9IDAuMCwgMAogICAgICAgIGZvciB4IGluIGxvYWRlcjoKICAgICAgICAgICAgeCA9IHgudG8oZGV2aWNlKQogICAgICAgICAgICAjIERlbm9pc2luZyBvYmplY3RpdmU6IHJlY29uc3RydWN0IHRoZSBDTEVBTiBpbWFnZSBmcm9tIGEgbm9pc2VkCiAgICAgICAgICAgICMgaW5wdXQuIEFub3RoZXIgYnJha2Ugb24gdGhlIGlkZW50aXR5IHNob3J0Y3V0IC0tIGNvcHlpbmcgdGhlCiAgICAgICAgICAgICMgaW5wdXQgdGhyb3VnaCBub3cgcmVwcm9kdWNlcyB0aGUgbm9pc2UgdG9vLCBhbmQgaXMgcHVuaXNoZWQuCiAgICAgICAgICAgIG5vaXNlID0gdG9yY2gucmFuZG5fbGlrZSh4KSAqIGNmZy5ub2lzZV9zaWdtYQogICAgICAgICAgICByZWNvbiA9IG1vZGVsKHggKyBub2lzZSkKICAgICAgICAgICAgbG9zcyA9IHJlY29uX2xvc3MocmVjb24sIHgsIGFscGhhPWNmZy5sb3NzX2FscGhhKQogICAgICAgICAgICBvcHQuemVyb19ncmFkKCkKICAgICAgICAgICAgbG9zcy5iYWNrd2FyZCgpCiAgICAgICAgICAgIG9wdC5zdGVwKCkKICAgICAgICAgICAgdG90YWwgKz0gbG9zcy5pdGVtKCkKICAgICAgICAgICAgbmIgKz0gMQogICAgICAgIGVwb2NoX2xvc3MgPSB0b3RhbCAvIG1heChuYiwgMSkKICAgICAgICBoaXN0b3J5LmFwcGVuZCh7ImVwb2NoIjogZXAgKyAxLCAibG9zcyI6IGVwb2NoX2xvc3N9KQogICAgICAgIGlmIHZlcmJvc2UgYW5kICgoZXAgKyAxKSAlIGxvZ19ldmVyeSA9PSAwIG9yIGVwID09IDApOgogICAgICAgICAgICBwcmludChmImVwb2NoIHtlcCArIDE6M2R9L3tjZmcuZXBvY2hzfSAgbG9zcyB7ZXBvY2hfbG9zczouNGZ9IiwgZmx1c2g9VHJ1ZSkKICAgIHJldHVybiBtb2RlbCwgaGlzdG9yeQoKCmRlZiBzYXZlX2NoZWNrcG9pbnQocGF0aCwgbW9kZWwsIGNmZzogQ29uZmlnLCBoaXN0b3J5LCBleHRyYT1Ob25lKSAtPiBQYXRoOgogICAgcGF0aCA9IFBhdGgocGF0aCkKICAgIHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHRvcmNoLnNhdmUoCiAgICAgICAgewogICAgICAgICAgICAic3RhdGVfZGljdCI6IG1vZGVsLnN0YXRlX2RpY3QoKSwKICAgICAgICAgICAgImNvbmZpZyI6IGNmZy50b19kaWN0KCksCiAgICAgICAgICAgICJoaXN0b3J5IjogaGlzdG9yeSwKICAgICAgICAgICAgInZlcnNpb24iOiBfX3ZlcnNpb25fXywKICAgICAgICAgICAgKiooZXh0cmEgb3Ige30pLAogICAgICAgIH0sCiAgICAgICAgcGF0aCwKICAgICkKICAgIHJldHVybiBwYXRoCgoKZGVmIGxvYWRfY2hlY2twb2ludChwYXRoLCBkZXZpY2U6IHRvcmNoLmRldmljZSk6CiAgICAiIiJSZWJ1aWxkIGEgbW9kZWwgZnJvbSBhIGNoZWNrcG9pbnQsIHVzaW5nIHRoZSBjb25maWcgc3RvcmVkIGluc2lkZSBpdC4iIiIKICAgIGNrcHQgPSB0b3JjaC5sb2FkKHBhdGgsIG1hcF9sb2NhdGlvbj1kZXZpY2UsIHdlaWdodHNfb25seT1GYWxzZSkKICAgIGNmZyA9IENvbmZpZygqKmNrcHRbImNvbmZpZyJdKQogICAgbW9kZWwgPSBDb252QXV0b2VuY29kZXIobGF0ZW50X2NoPWNmZy5sYXRlbnRfY2gpLnRvKGRldmljZSkKICAgIG1vZGVsLmxvYWRfc3RhdGVfZGljdChja3B0WyJzdGF0ZV9kaWN0Il0pCiAgICBtb2RlbC5ldmFsKCkKICAgIHJldHVybiBtb2RlbCwgY2ZnLCBja3B0CgoKZGVmIGJ1aWxkX3BhcnNlcigpIC0+IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyOgogICAgcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPSJUcmFpbiB0aGUgYm90dGxlLWRlZmVjdCBhdXRvZW5jb2Rlci4iKQogICAgZCA9IENvbmZpZygpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1kYXRhLXJvb3QiLCByZXF1aXJlZD1UcnVlLCBoZWxwPSJNVlRlYyBjYXRlZ29yeSBmb2xkZXIgKGhhcyB0cmFpbi9nb29kKSIpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1vdXQiLCBkZWZhdWx0PSJjaGVja3BvaW50cy9hZV92My5wdCIsIGhlbHA9ImNoZWNrcG9pbnQgcGF0aCIpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1lcG9jaHMiLCB0eXBlPWludCwgZGVmYXVsdD1kLmVwb2NocykKICAgIHAuYWRkX2FyZ3VtZW50KCItLWJhdGNoLXNpemUiLCB0eXBlPWludCwgZGVmYXVsdD1kLmJhdGNoX3NpemUpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1sciIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9ZC5scikKICAgIHAuYWRkX2FyZ3VtZW50KCItLWltZy1zaXplIiwgdHlwZT1pbnQsIGRlZmF1bHQ9ZC5pbWdfc2l6ZSkKICAgIHAuYWRkX2FyZ3VtZW50KCItLWxhdGVudC1jaCIsIHR5cGU9aW50LCBkZWZhdWx0PWQubGF0ZW50X2NoKQogICAgcC5hZGRfYXJndW1lbnQoIi0tbm9pc2Utc2lnbWEiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PWQubm9pc2Vfc2lnbWEpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1zZWVkIiwgdHlwZT1pbnQsIGRlZmF1bHQ9ZC5zZWVkKQogICAgcC5hZGRfYXJndW1lbnQoIi0tbnVtLXdvcmtlcnMiLCB0eXBlPWludCwgZGVmYXVsdD1kLm51bV93b3JrZXJzKQogICAgcC5hZGRfYXJndW1lbnQoIi0tbG9nLWV2ZXJ5IiwgdHlwZT1pbnQsIGRlZmF1bHQ9MTAsIGhlbHA9InByaW50IGxvc3MgZXZlcnkgTiBlcG9jaHMiKQogICAgcC5hZGRfYXJndW1lbnQoIi0tZGV2aWNlIiwgZGVmYXVsdD0iYXV0byIsIGNob2ljZXM9WyJhdXRvIiwgImNwdSIsICJjdWRhIl0pCiAgICByZXR1cm4gcAoKCmRlZiBtYWluKGFyZ3Y9Tm9uZSkgLT4gaW50OgogICAgYXJncyA9IGJ1aWxkX3BhcnNlcigpLnBhcnNlX2FyZ3MoYXJndikKICAgIGNmZyA9IENvbmZpZygKICAgICAgICBpbWdfc2l6ZT1hcmdzLmltZ19zaXplLAogICAgICAgIG51bV93b3JrZXJzPWFyZ3MubnVtX3dvcmtlcnMsCiAgICAgICAgbGF0ZW50X2NoPWFyZ3MubGF0ZW50X2NoLAogICAgICAgIGVwb2Nocz1hcmdzLmVwb2NocywKICAgICAgICBiYXRjaF9zaXplPWFyZ3MuYmF0Y2hfc2l6ZSwKICAgICAgICBscj1hcmdzLmxyLAogICAgICAgIG5vaXNlX3NpZ21hPWFyZ3Mubm9pc2Vfc2lnbWEsCiAgICAgICAgc2VlZD1hcmdzLnNlZWQsCiAgICApCiAgICBkZXZpY2UgPSBwaWNrX2RldmljZShhcmdzLmRldmljZSkKICAgIHByaW50KGYiZGV2aWNlOiB7ZGV2aWNlfSAgfCAgY29uZmlnOiB7anNvbi5kdW1wcyhjZmcudG9fZGljdCgpKX0iLCBmbHVzaD1UcnVlKQoKICAgIHQwID0gdGltZS50aW1lKCkKICAgIG1vZGVsLCBoaXN0b3J5ID0gdHJhaW4oYXJncy5kYXRhX3Jvb3QsIGNmZywgZGV2aWNlLCBsb2dfZXZlcnk9YXJncy5sb2dfZXZlcnkpCiAgICBlbGFwc2VkID0gdGltZS50aW1lKCkgLSB0MAoKICAgIG91dCA9IHNhdmVfY2hlY2twb2ludCgKICAgICAgICBhcmdzLm91dCwgbW9kZWwsIGNmZywgaGlzdG9yeSwKICAgICAgICBleHRyYT17ImRhdGFfcm9vdCI6IHN0cihhcmdzLmRhdGFfcm9vdCksICJ0cmFpbl9zZWNvbmRzIjogcm91bmQoZWxhcHNlZCwgMSksCiAgICAgICAgICAgICAgICJkZXZpY2UiOiBzdHIoZGV2aWNlKX0sCiAgICApCiAgICBwcmludChmIlxudHJhaW5lZCBpbiB7ZWxhcHNlZDouMWZ9cyAgLT4gIHtvdXR9IikKICAgIHJldHVybiAwCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHJhaXNlIFN5c3RlbUV4aXQobWFpbigpKQo=',
    'defectloc/evaluate.py': 'IiIiRXZhbHVhdGUgYSBjaGVja3BvaW50OiBpbWFnZS1BVVJPQyBhbmQgcGl4ZWwtQVVST0MsIFBhdGNoQ29yZS1jb21wYXJhYmxlLgoKU2NvcmVzIGJvdGggd2F5cyBpbiBhIHNpbmdsZSBwYXNzIG92ZXIgdGhlIHRlc3Qgc2V0IC0tIHJhdyB0b3AtayAodGhlIHYzCm5vdGVib29rIGJlaGF2aW91cikgYW5kIG1lZGlhbi9NQUQtbm9ybWFsaXNlZCAodGhlIGZpeCkgLS0gc28gdGhlIHR3byBhcmUKYWx3YXlzIG1lYXN1cmVkIG9uIGlkZW50aWNhbCBtYXBzIGFuZCB0aGUgY29tcGFyaXNvbiBjYW5ub3QgZHJpZnQuCgogICAgcHl0aG9uIC1tIGRlZmVjdGxvYy5ldmFsdWF0ZSAtLWNoZWNrcG9pbnQgY2hlY2twb2ludHMvYWVfdjMucHQgXAogICAgICAgIC0tY2xlYW4tcm9vdCBkYXRhL01WVGVjQUQvYm90dGxlIFwKICAgICAgICAtLWxpZ2h0aW5nLXJvb3QgZGF0YS9NVlRlY0FEX2xpZ2h0aW5nL2JvdHRsZQoiIiIKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQganNvbgpmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBkZWZhdWx0ZGljdApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKZnJvbSBza2xlYXJuLm1ldHJpY3MgaW1wb3J0IHJvY19hdWNfc2NvcmUKZnJvbSB0b3JjaC51dGlscy5kYXRhIGltcG9ydCBEYXRhTG9hZGVyCgpmcm9tIGRlZmVjdGxvYy5hbm9tYWx5IGltcG9ydCBhbm9tYWx5X21hcCwgaW1hZ2Vfc2NvcmUsIG5vcm1hbGl6ZV9tYXAKZnJvbSBkZWZlY3Rsb2MuY29uZmlnIGltcG9ydCBDb25maWcKZnJvbSBkZWZlY3Rsb2MuZGF0YSBpbXBvcnQgVGVzdFNldApmcm9tIGRlZmVjdGxvYy50cmFpbiBpbXBvcnQgbG9hZF9jaGVja3BvaW50LCBwaWNrX2RldmljZSwgc2V0X3NlZWQKClNDT1JFX1ZBUklBTlRTID0gKCJub25lIiwgIm1lZGlhbl9tYWQiKQoKCmRlZiBfYXVyb2MoeV90cnVlLCB5X3Njb3JlKToKICAgICIiIkFVUk9DLCBvciBOb25lIHdoZW4gb25seSBvbmUgY2xhc3MgaXMgcHJlc2VudCAoYW4gdW5kZWZpbmVkIG1ldHJpYykuIiIiCiAgICB5X3RydWUgPSBucC5hc2FycmF5KHlfdHJ1ZSkKICAgIGlmIGxlbihucC51bmlxdWUoeV90cnVlKSkgPCAyOgogICAgICAgIHJldHVybiBOb25lCiAgICByZXR1cm4gZmxvYXQocm9jX2F1Y19zY29yZSh5X3RydWUsIHlfc2NvcmUpKQoKCkB0b3JjaC5ub19ncmFkKCkKZGVmIGV2YWx1YXRlKG1vZGVsLCByb290LCBjZmc6IENvbmZpZywgZGV2aWNlLCBiYXRjaF9zaXplOiBpbnQgPSA4KSAtPiBkaWN0OgogICAgIiIiUmV0dXJuIGltYWdlL3BpeGVsIEFVUk9DIGZvciBldmVyeSBzY29yaW5nIHZhcmlhbnQsIHBsdXMgYSBwZXItdHlwZSBicmVha2Rvd24uIiIiCiAgICBkcyA9IFRlc3RTZXQocm9vdCwgaW1nX3NpemU9Y2ZnLmltZ19zaXplKQogICAgbG9hZGVyID0gRGF0YUxvYWRlcihkcywgYmF0Y2hfc2l6ZT1iYXRjaF9zaXplLCBzaHVmZmxlPUZhbHNlLCBudW1fd29ya2Vycz1jZmcubnVtX3dvcmtlcnMpCgogICAgIyBEZWZlY3QgdHlwZSBwZXIgaXRlbSwgaW4gZGF0YXNldCBvcmRlciwgZm9yIHRoZSBwZXItdHlwZSBicmVha2Rvd24uCiAgICB0eXBlcyA9IFtwLnBhcmVudC5uYW1lIGZvciBwLCBfLCBfIGluIGRzLml0ZW1zXQoKICAgIHBpeGVsX2d0ID0gW10KICAgIHBpeGVsX3Njb3JlcyA9IHt2OiBbXSBmb3IgdiBpbiBTQ09SRV9WQVJJQU5UU30KICAgIGltZ19ndCA9IFtdCiAgICBpbWdfc2NvcmVzID0ge3Y6IFtdIGZvciB2IGluIFNDT1JFX1ZBUklBTlRTfQoKICAgIGZvciBpbWdzLCBtYXNrcywgbGFiZWxzIGluIGxvYWRlcjoKICAgICAgICBhbWFwcyA9IGFub21hbHlfbWFwKG1vZGVsLCBpbWdzLnRvKGRldmljZSksIHdpbj1jZmcuc3NpbV93aW5kb3cpICAjIEIsSCxXCiAgICAgICAgbWFza3MgPSBtYXNrcy5udW1weSgpCiAgICAgICAgZm9yIGFtYXAsIG0sIGxhYiBpbiB6aXAoYW1hcHMsIG1hc2tzLCBsYWJlbHMubnVtcHkoKSk6CiAgICAgICAgICAgIHBpeGVsX2d0LmFwcGVuZChtLnJhdmVsKCkpCiAgICAgICAgICAgIGltZ19ndC5hcHBlbmQoaW50KGxhYikpCiAgICAgICAgICAgIGZvciB2IGluIFNDT1JFX1ZBUklBTlRTOgogICAgICAgICAgICAgICAgc2NvcmVkID0gbm9ybWFsaXplX21hcChhbWFwKSBpZiB2ID09ICJtZWRpYW5fbWFkIiBlbHNlIGFtYXAKICAgICAgICAgICAgICAgIHBpeGVsX3Njb3Jlc1t2XS5hcHBlbmQoc2NvcmVkLnJhdmVsKCkpCiAgICAgICAgICAgICAgICBpbWdfc2NvcmVzW3ZdLmFwcGVuZChpbWFnZV9zY29yZShhbWFwLCB0b3BrX2ZyYWM9Y2ZnLnRvcGtfZnJhYywgbm9ybT12KSkKCiAgICBwaXhlbF9ndF9hbGwgPSBucC5jb25jYXRlbmF0ZShwaXhlbF9ndCkKICAgIGltZ19ndCA9IG5wLmFzYXJyYXkoaW1nX2d0KQogICAgdHlwZXMgPSBucC5hc2FycmF5KHR5cGVzKQoKICAgIG91dCA9IHsibl9pbWFnZXMiOiBsZW4oZHMpLCAibl9kZWZlY3RpdmUiOiBpbnQoaW1nX2d0LnN1bSgpKSwgInZhcmlhbnRzIjoge319CiAgICBmb3IgdiBpbiBTQ09SRV9WQVJJQU5UUzoKICAgICAgICBpbWdfcyA9IG5wLmFzYXJyYXkoaW1nX3Njb3Jlc1t2XSkKICAgICAgICBweF9zID0gbnAuY29uY2F0ZW5hdGUocGl4ZWxfc2NvcmVzW3ZdKQoKICAgICAgICBwZXJfdHlwZSA9IHt9CiAgICAgICAgZ29vZCA9IHR5cGVzID09ICJnb29kIgogICAgICAgIGZvciB0IGluIHNvcnRlZChzZXQodHlwZXNbfmdvb2RdKSk6CiAgICAgICAgICAgIHNlbCA9IGdvb2QgfCAodHlwZXMgPT0gdCkKICAgICAgICAgICAgcHhfc2VsID0gbnAuY29uY2F0ZW5hdGUoW3BpeGVsX3Njb3Jlc1t2XVtpXSBmb3IgaSBpbiBucC5mbGF0bm9uemVybyhzZWwpXSkKICAgICAgICAgICAgZ3Rfc2VsID0gbnAuY29uY2F0ZW5hdGUoW3BpeGVsX2d0W2ldIGZvciBpIGluIG5wLmZsYXRub256ZXJvKHNlbCldKQogICAgICAgICAgICBwZXJfdHlwZVt0XSA9IHsKICAgICAgICAgICAgICAgICJuIjogaW50KCh0eXBlcyA9PSB0KS5zdW0oKSksCiAgICAgICAgICAgICAgICAiaW1hZ2VfQVVST0MiOiBfYXVyb2MoaW1nX2d0W3NlbF0sIGltZ19zW3NlbF0pLAogICAgICAgICAgICAgICAgInBpeGVsX0FVUk9DIjogX2F1cm9jKGd0X3NlbCwgcHhfc2VsKSwKICAgICAgICAgICAgfQoKICAgICAgICBvdXRbInZhcmlhbnRzIl1bdl0gPSB7CiAgICAgICAgICAgICJpbWFnZV9BVVJPQyI6IF9hdXJvYyhpbWdfZ3QsIGltZ19zKSwKICAgICAgICAgICAgInBpeGVsX0FVUk9DIjogX2F1cm9jKHBpeGVsX2d0X2FsbCwgcHhfcyksCiAgICAgICAgICAgICJwZXJfZGVmZWN0X3R5cGUiOiBwZXJfdHlwZSwKICAgICAgICB9CiAgICByZXR1cm4gb3V0CgoKZGVmIGJ1aWxkX3BhcnNlcigpIC0+IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyOgogICAgcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPSJFdmFsdWF0ZSBhIGJvdHRsZS1kZWZlY3QgYXV0b2VuY29kZXIgY2hlY2twb2ludC4iKQogICAgcC5hZGRfYXJndW1lbnQoIi0tY2hlY2twb2ludCIsIHJlcXVpcmVkPVRydWUpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1jbGVhbi1yb290IiwgcmVxdWlyZWQ9VHJ1ZSwgaGVscD0iY2xlYW4gTVZUZWMgY2F0ZWdvcnkgZm9sZGVyIikKICAgIHAuYWRkX2FyZ3VtZW50KCItLWxpZ2h0aW5nLXJvb3QiLCBkZWZhdWx0PU5vbmUsIGhlbHA9ImxpZ2h0aW5nLXN0cmVzc2VkIGNvcHkgKG9wdGlvbmFsKSIpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1vdXQiLCBkZWZhdWx0PSJyZXN1bHRzL3J1bnMvbGF0ZXN0Lmpzb24iLCBoZWxwPSJ3aGVyZSB0byB3cml0ZSB0aGUgcnVuIEpTT04iKQogICAgcC5hZGRfYXJndW1lbnQoIi0tbGFiZWwiLCBkZWZhdWx0PSJBdXRvZW5jb2RlciB2MyIsIGhlbHA9InJvdyBsYWJlbCBpbiB0aGUgcmVzdWx0cyB0YWJsZSIpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1iYXRjaC1zaXplIiwgdHlwZT1pbnQsIGRlZmF1bHQ9OCkKICAgIHAuYWRkX2FyZ3VtZW50KCItLWRldmljZSIsIGRlZmF1bHQ9ImF1dG8iLCBjaG9pY2VzPVsiYXV0byIsICJjcHUiLCAiY3VkYSJdKQogICAgcmV0dXJuIHAKCgpkZWYgbWFpbihhcmd2PU5vbmUpIC0+IGludDoKICAgIGFyZ3MgPSBidWlsZF9wYXJzZXIoKS5wYXJzZV9hcmdzKGFyZ3YpCiAgICBkZXZpY2UgPSBwaWNrX2RldmljZShhcmdzLmRldmljZSkKICAgIG1vZGVsLCBjZmcsIGNrcHQgPSBsb2FkX2NoZWNrcG9pbnQoYXJncy5jaGVja3BvaW50LCBkZXZpY2UpCiAgICBzZXRfc2VlZChjZmcuc2VlZCkKICAgIHByaW50KGYiZGV2aWNlOiB7ZGV2aWNlfSAgfCAgY2hlY2twb2ludDoge2FyZ3MuY2hlY2twb2ludH0iLCBmbHVzaD1UcnVlKQoKICAgIHJ1biA9IHsKICAgICAgICAibGFiZWwiOiBhcmdzLmxhYmVsLAogICAgICAgICJjaGVja3BvaW50Ijogc3RyKGFyZ3MuY2hlY2twb2ludCksCiAgICAgICAgImNvbmZpZyI6IGNmZy50b19kaWN0KCksCiAgICAgICAgInRyYWluX3NlY29uZHMiOiBja3B0LmdldCgidHJhaW5fc2Vjb25kcyIpLAogICAgICAgICJmaW5hbF90cmFpbl9sb3NzIjogKGNrcHQuZ2V0KCJoaXN0b3J5Iikgb3IgW3t9XSlbLTFdLmdldCgibG9zcyIpLAogICAgICAgICJjb25kaXRpb25zIjoge30sCiAgICB9CgogICAgZm9yIG5hbWUsIHJvb3QgaW4gKCgiY2xlYW4iLCBhcmdzLmNsZWFuX3Jvb3QpLCAoImxpZ2h0aW5nIiwgYXJncy5saWdodGluZ19yb290KSk6CiAgICAgICAgaWYgcm9vdCBpcyBOb25lOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHJlcyA9IGV2YWx1YXRlKG1vZGVsLCByb290LCBjZmcsIGRldmljZSwgYmF0Y2hfc2l6ZT1hcmdzLmJhdGNoX3NpemUpCiAgICAgICAgcmVzWyJyb290Il0gPSBzdHIocm9vdCkKICAgICAgICBydW5bImNvbmRpdGlvbnMiXVtuYW1lXSA9IHJlcwogICAgICAgIGZvciB2IGluIFNDT1JFX1ZBUklBTlRTOgogICAgICAgICAgICBtID0gcmVzWyJ2YXJpYW50cyJdW3ZdCiAgICAgICAgICAgIHByaW50KAogICAgICAgICAgICAgICAgZiJ7bmFtZTo5c30gW3t2OjEwc31dICBpbWFnZS1BVVJPQyB7bVsnaW1hZ2VfQVVST0MnXTouNGZ9IgogICAgICAgICAgICAgICAgZiIgICBwaXhlbC1BVVJPQyB7bVsncGl4ZWxfQVVST0MnXTouNGZ9IgogICAgICAgICAgICApCgogICAgb3V0ID0gUGF0aChhcmdzLm91dCkKICAgIG91dC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgb3V0LndyaXRlX3RleHQoanNvbi5kdW1wcyhydW4sIGluZGVudD0yKSwgZW5jb2Rpbmc9InV0Zi04IikKICAgIHByaW50KGYiXG53cm90ZSB7b3V0fSIpCiAgICByZXR1cm4gMAoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICByYWlzZSBTeXN0ZW1FeGl0KG1haW4oKSkK',
    'defectloc/visualize.py': 'IiIiUXVhbGl0YXRpdmUgY2hlY2s6IGlucHV0IC8gcmVjb25zdHJ1Y3Rpb24gLyBhbm9tYWx5IG1hcCAvIGdyb3VuZCB0cnV0aC4KCktlcHQgYXMgYSBmaXJzdC1jbGFzcyBlbnRyeSBwb2ludCByYXRoZXIgdGhhbiBhIG5vdGVib29rIGNlbGwgYmVjYXVzZSBsb29raW5nCmF0IHRoZSBtYXBzIGJlZm9yZSB0cnVzdGluZyBhIG1ldHJpYyBpcyB0aGUgZGlzY2lwbGluZSB0aGF0IHByb2R1Y2VkIHYzLiBBCnBpeGVsLUFVUk9DIGNhbiBsb29rIHJlc3BlY3RhYmxlIHdoaWxlIHRoZSBtYXAgZmlyZXMgb24gcmltcyBhbmQgcmVmbGVjdGlvbnMKaW5zdGVhZCBvZiB0aGUgZGVmZWN0OyBvbmx5IHRoZSBwaWN0dXJlIHNob3dzIHRoYXQuCgogICAgcHl0aG9uIC1tIGRlZmVjdGxvYy52aXN1YWxpemUgLS1jaGVja3BvaW50IGNoZWNrcG9pbnRzL2FlX3YzLnB0IFwKICAgICAgICAtLWRhdGEtcm9vdCBkYXRhL01WVGVjQUQvYm90dGxlIC0tb3V0IHJlc3VsdHMvZmlndXJlcy92M19jbGVhbi5wbmcKIiIiCgppbXBvcnQgYXJncGFyc2UKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgbWF0cGxvdGxpYgppbXBvcnQgdG9yY2gKCm1hdHBsb3RsaWIudXNlKCJBZ2ciKQppbXBvcnQgbWF0cGxvdGxpYi5weXBsb3QgYXMgcGx0ICAjIG5vcWE6IEU0MDIKCmZyb20gZGVmZWN0bG9jLmFub21hbHkgaW1wb3J0IGFub21hbHlfbWFwLCBub3JtYWxpemVfbWFwICAjIG5vcWE6IEU0MDIKZnJvbSBkZWZlY3Rsb2MuZGF0YSBpbXBvcnQgVGVzdFNldCAgIyBub3FhOiBFNDAyCmZyb20gZGVmZWN0bG9jLnRyYWluIGltcG9ydCBsb2FkX2NoZWNrcG9pbnQsIHBpY2tfZGV2aWNlICAjIG5vcWE6IEU0MDIKCgpAdG9yY2gubm9fZ3JhZCgpCmRlZiBtYWtlX2ZpZ3VyZShtb2RlbCwgY2ZnLCByb290LCBkZXZpY2UsIG46IGludCA9IDMsIG91dD1Ob25lLCBub3JtYWxpemVkOiBib29sID0gRmFsc2UpOgogICAgZHMgPSBUZXN0U2V0KHJvb3QsIGltZ19zaXplPWNmZy5pbWdfc2l6ZSkKICAgIGRlZmVjdGl2ZSA9IFtpIGZvciBpLCAoXywgXywgbGFiKSBpbiBlbnVtZXJhdGUoZHMuaXRlbXMpIGlmIGxhYiA9PSAxXVs6bl0KICAgIGlmIG5vdCBkZWZlY3RpdmU6CiAgICAgICAgcmFpc2UgU3lzdGVtRXhpdChmIntyb290fSBoYXMgbm8gZGVmZWN0aXZlIHRlc3QgaW1hZ2VzIHRvIHZpc3VhbGlzZSIpCgogICAgZmlnLCBheGVzID0gcGx0LnN1YnBsb3RzKGxlbihkZWZlY3RpdmUpLCA0LCBmaWdzaXplPSgxNCwgMy40ICogbGVuKGRlZmVjdGl2ZSkpKQogICAgYXhlcyA9IGF4ZXMucmVzaGFwZShsZW4oZGVmZWN0aXZlKSwgNCkKCiAgICBmb3Igcm93LCBpZHggaW4gZW51bWVyYXRlKGRlZmVjdGl2ZSk6CiAgICAgICAgaW1nLCBtYXNrLCBfID0gZHNbaWR4XQogICAgICAgIHggPSBpbWcudW5zcXVlZXplKDApLnRvKGRldmljZSkKICAgICAgICByZWNvbiA9IG1vZGVsKHgpWzBdLmNwdSgpLnBlcm11dGUoMSwgMiwgMCkubnVtcHkoKQogICAgICAgIGFtYXAgPSBhbm9tYWx5X21hcChtb2RlbCwgeCwgd2luPWNmZy5zc2ltX3dpbmRvdylbMF0KICAgICAgICBpZiBub3JtYWxpemVkOgogICAgICAgICAgICBhbWFwID0gbm9ybWFsaXplX21hcChhbWFwKQoKICAgICAgICBkZWZlY3RfdHlwZSA9IGRzLml0ZW1zW2lkeF1bMF0ucGFyZW50Lm5hbWUKICAgICAgICBwYW5lbHMgPSBbCiAgICAgICAgICAgIChpbWcucGVybXV0ZSgxLCAyLCAwKS5udW1weSgpLCBmImlucHV0ICh7ZGVmZWN0X3R5cGV9KSIsIE5vbmUpLAogICAgICAgICAgICAocmVjb24sICJyZWNvbnN0cnVjdGlvbiIsIE5vbmUpLAogICAgICAgICAgICAoYW1hcCwgImFub21hbHkgbWFwIiArICgiIChtZWRpYW4vTUFEKSIgaWYgbm9ybWFsaXplZCBlbHNlICIiKSwgImluZmVybm8iKSwKICAgICAgICAgICAgKG1hc2subnVtcHkoKSwgImdyb3VuZCB0cnV0aCIsICJncmF5IiksCiAgICAgICAgXQogICAgICAgIGZvciBjb2wsIChkYXRhLCB0aXRsZSwgY21hcCkgaW4gZW51bWVyYXRlKHBhbmVscyk6CiAgICAgICAgICAgIGF4ID0gYXhlc1tyb3csIGNvbF0KICAgICAgICAgICAgYXguaW1zaG93KGRhdGEsIGNtYXA9Y21hcCkKICAgICAgICAgICAgYXguc2V0X3RpdGxlKHRpdGxlLCBmb250c2l6ZT0xMCkKICAgICAgICAgICAgYXguYXhpcygib2ZmIikKCiAgICBmaWcudGlnaHRfbGF5b3V0KCkKICAgIGlmIG91dDoKICAgICAgICBQYXRoKG91dCkucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICBmaWcuc2F2ZWZpZyhvdXQsIGRwaT0xMzAsIGJib3hfaW5jaGVzPSJ0aWdodCIpCiAgICAgICAgcHJpbnQoZiJ3cm90ZSB7b3V0fSIpCiAgICBwbHQuY2xvc2UoZmlnKQogICAgcmV0dXJuIG91dAoKCmRlZiBtYWluKGFyZ3Y9Tm9uZSkgLT4gaW50OgogICAgcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPV9fZG9jX18uc3BsaXRsaW5lcygpWzBdKQogICAgcC5hZGRfYXJndW1lbnQoIi0tY2hlY2twb2ludCIsIHJlcXVpcmVkPVRydWUpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1kYXRhLXJvb3QiLCByZXF1aXJlZD1UcnVlKQogICAgcC5hZGRfYXJndW1lbnQoIi0tb3V0IiwgZGVmYXVsdD0icmVzdWx0cy9maWd1cmVzL2Fub21hbHlfbWFwcy5wbmciKQogICAgcC5hZGRfYXJndW1lbnQoIi0tbiIsIHR5cGU9aW50LCBkZWZhdWx0PTMpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1ub3JtYWxpemVkIiwgYWN0aW9uPSJzdG9yZV90cnVlIiwgaGVscD0ic2hvdyB0aGUgbWVkaWFuL01BRC1zY2FsZWQgbWFwIikKICAgIHAuYWRkX2FyZ3VtZW50KCItLWRldmljZSIsIGRlZmF1bHQ9ImF1dG8iLCBjaG9pY2VzPVsiYXV0byIsICJjcHUiLCAiY3VkYSJdKQogICAgYSA9IHAucGFyc2VfYXJncyhhcmd2KQoKICAgIGRldmljZSA9IHBpY2tfZGV2aWNlKGEuZGV2aWNlKQogICAgbW9kZWwsIGNmZywgXyA9IGxvYWRfY2hlY2twb2ludChhLmNoZWNrcG9pbnQsIGRldmljZSkKICAgIG1ha2VfZmlndXJlKG1vZGVsLCBjZmcsIGEuZGF0YV9yb290LCBkZXZpY2UsIG49YS5uLCBvdXQ9YS5vdXQsIG5vcm1hbGl6ZWQ9YS5ub3JtYWxpemVkKQogICAgcmV0dXJuIDAKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgcmFpc2UgU3lzdGVtRXhpdChtYWluKCkpCg==',
}
for path, blob in FILES.items():
    p = pathlib.Path('/kaggle/working') / path
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(base64.b64decode(blob))
sys.path.insert(0, '/kaggle/working')
print('wrote', len(FILES), 'modules')


## 2. Locate the dataset

Searches the attached inputs instead of hardcoding a path. The hardcoded
`DATA_ROOT` is what broke the original notebook when the mount point changed.


In [ ]:
import pathlib
CATEGORY = 'bottle'

hits = sorted(pathlib.Path('/kaggle/input').glob(f'**/{CATEGORY}/train/good'))
if not hits:
    raise SystemExit(
        f'no {CATEGORY}/train/good under /kaggle/input.\n'
        'Attach the ipythonx/mvtec-ad dataset via + Add Input.'
    )
# hits[0] is <root>/<category>/train/good, so the category dir is two levels up.
CAT_DIR = hits[0].parents[1]
SRC_ROOT = CAT_DIR.parent
assert CAT_DIR.name == CATEGORY, CAT_DIR
print('category dir:', CAT_DIR)
print('contains: ', sorted(p.name for p in CAT_DIR.iterdir() if p.is_dir()))
print('siblings:  ', sorted(p.name for p in SRC_ROOT.iterdir() if p.is_dir())[:20])


## 3. Build the clean and lighting-stressed roots

The lighting stress test is seeded, so this is the same set of images every run.


In [ ]:
import shutil, cv2, pathlib
from defectloc.augment import build_factory_lighting

SEED = 0
# /kaggle/temp, not /kaggle/working: everything under working becomes kernel
# output, so staging the dataset there makes `kaggle kernels output` drag back
# hundreds of megabytes of images to retrieve a few kilobytes of results.
CLEAN = pathlib.Path('/kaggle/temp/data/MVTecAD') / CATEGORY
LIGHT = pathlib.Path('/kaggle/temp/data/MVTecAD_lighting') / CATEGORY

if not CLEAN.exists():
    shutil.copytree(CAT_DIR, CLEAN)
print('clean:', CLEAN, '| train imgs:', len(list((CLEAN / 'train' / 'good').glob('*.png'))))

if not LIGHT.exists():
    lighting = build_factory_lighting(seed=SEED)
    n = 0
    for split in sorted((CLEAN / 'test').iterdir()):
        if not split.is_dir():
            continue
        for img_path in sorted(split.glob('*.png')):
            img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
            aug = cv2.cvtColor(lighting(image=img)['image'], cv2.COLOR_RGB2BGR)
            out = LIGHT / 'test' / split.name / img_path.name
            out.parent.mkdir(parents=True, exist_ok=True)
            cv2.imwrite(str(out), aug)
            n += 1
    shutil.copytree(CLEAN / 'ground_truth', LIGHT / 'ground_truth', dirs_exist_ok=True)
    shutil.copytree(CLEAN / 'train', LIGHT / 'train', dirs_exist_ok=True)
    print(f'lighting: {LIGHT} ({n} test images stressed, seed={SEED})')


## 4. Train

Same architecture, loss and objective as v3.


In [ ]:
import time
from defectloc.config import Config
from defectloc.train import train, save_checkpoint, pick_device

cfg = Config(epochs=80, seed=SEED, num_workers=2)
# pick_device probes the GPU with a real conv first. Kaggle sometimes assigns a
# Tesla P100 (sm_60) that the preinstalled torch cannot run, and reports it as
# available anyway; the probe catches that here instead of mid-training.
device = pick_device('auto')
print('training on:', device)
if device.type == 'cpu':
    print('WARNING: running on CPU. Expect roughly an hour rather than minutes.')
t0 = time.time()
model, history = train(CLEAN, cfg, device, log_every=10)
elapsed = time.time() - t0
ckpt = save_checkpoint('/kaggle/working/checkpoints/ae_v3.pt', model, cfg, history,
                       extra={'data_root': str(CLEAN), 'train_seconds': round(elapsed, 1),
                              'device': str(device)})
print(f'trained in {elapsed:.1f}s -> {ckpt}')


## 5. Look at the maps before reading the metric

A pixel-AUROC can look respectable while the map fires on rims and reflections
instead of the defect. Inspect first.


In [ ]:
from defectloc.visualize import make_figure
make_figure(model, cfg, CLEAN, device, n=3,
            out='/kaggle/working/results/figures/v3_clean.png')
make_figure(model, cfg, LIGHT, device, n=3,
            out='/kaggle/working/results/figures/v3_lighting.png')
from IPython.display import Image, display
display(Image('/kaggle/working/results/figures/v3_clean.png'))
display(Image('/kaggle/working/results/figures/v3_lighting.png'))


## 6. Evaluate

Both scoring variants, one pass, identical maps. `median_mad` under `lighting` is
the number the results table is missing.


In [ ]:
import json
from defectloc.evaluate import evaluate, SCORE_VARIANTS

run = {'label': 'Autoencoder v3', 'checkpoint': str(ckpt), 'config': cfg.to_dict(),
       'train_seconds': round(elapsed, 1),
       'final_train_loss': history[-1]['loss'],
       'environment': {'device': str(device), 'torch': torch.__version__,
                       'ran_on': 'kaggle'},
       'conditions': {}}

for name, root in (('clean', CLEAN), ('lighting', LIGHT)):
    res = evaluate(model, root, cfg, device, batch_size=8)
    res['root'] = str(root)
    run['conditions'][name] = res
    for v in SCORE_VARIANTS:
        m = res['variants'][v]
        print(f"{name:9s} [{v:10s}]  image-AUROC {m['image_AUROC']:.4f}"
              f"   pixel-AUROC {m['pixel_AUROC']:.4f}")

import pathlib
out = pathlib.Path('/kaggle/working/results/runs/ae_v3_kaggle.json')
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(run, indent=2))
print('\nwrote', out)


## 7. The row, and whether the fix worked


In [ ]:
raw_l = run['conditions']['lighting']['variants']['none']['image_AUROC']
nrm_l = run['conditions']['lighting']['variants']['median_mad']['image_AUROC']
raw_c = run['conditions']['clean']['variants']['none']['image_AUROC']
nrm_c = run['conditions']['clean']['variants']['median_mad']['image_AUROC']
print(f'lighting image-AUROC: raw {raw_l:.3f} -> median/MAD {nrm_l:.3f} '
      f'({nrm_l - raw_l:+.3f})')
print(f'clean    image-AUROC: raw {raw_c:.3f} -> median/MAD {nrm_c:.3f} '
      f'({nrm_c - raw_c:+.3f})')
print()
print('chance is 0.500. The fix is only a win if lighting image-AUROC rises well')
print('clear of chance WITHOUT costing much on clean. A clean drop is a real cost,')
print('not a rounding detail: report it either way.')
